# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ma5029blp-wq/ML-flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1.
**Method:** Logistic Regression.

This is a binary classification problem because the target tells us whether a content page shows an observed decline or not. Logistic Regression is a suitable first model because it is simple, interpretable, and produces a probability that can be used to rank pages for review.

I will compare the learned model against my Week-4 rule-based baseline. The goal is not to add complexity for its own sake, but to see whether the model can improve the baseline's precision when selecting pages for review.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2.

I will use a grouped train/test split by `client_id`. This keeps all pages from the same client in either the training set or the test set, rather than allowing the same client to appear in both.

This is an honest split because client-specific patterns could otherwise make the model look stronger than it really is. I will use a fixed random seed so the split can be reproduced.

In [4]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

overlap = set(train_df["client_id"]) & set(test_df["client_id"])

print("\nClient overlap:", len(overlap))

Train shape: (22885, 44)
Test shape: (7115, 44)

Train clients: 24
Test clients: 8

Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3.  

I trained a Logistic Regression model because this is a binary decline question and Logistic Regression provides a simple, interpretable first learned model.

The model was evaluated on the same client-grouped test set as the Week-4 baseline. At Precision@20, the Logistic Regression scored 0.50 compared with 0.25 for the baseline. At Precision@50, it scored 0.62 compared with 0.44 for the baseline. The model therefore performed better than the baseline at both ranking cutoffs.

The test-set base rate was 0.517. The model was above the base rate at Precision@50, while its Precision@20 was slightly below the base rate. This suggests the model provides useful ranking improvement at 50 items, but its very top 20 predictions should be interpreted cautiously.

In [5]:
train_df["declining_observed"] = (
    train_df["impressions_last_30d"]
    < 0.8 * train_df["impressions_prev_30d"]
).astype(int)

test_df["declining_observed"] = (
    test_df["impressions_last_30d"]
    < 0.8 * test_df["impressions_prev_30d"]
).astype(int)

print("Train decline rate:", train_df["declining_observed"].mean().round(4))
print("Test decline rate:", test_df["declining_observed"].mean().round(4))

print("\nTrain label counts:")
print(train_df["declining_observed"].value_counts())

print("\nTest label counts:")
print(test_df["declining_observed"].value_counts())

Train decline rate: 0.55
Test decline rate: 0.5165

Train label counts:
declining_observed
1    12587
0    10298
Name: count, dtype: int64

Test label counts:
declining_observed
1    3675
0    3440
Name: count, dtype: int64


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric_features = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "days_since_last_update",
    "content_age_days",
    "avg_position",
    "search_volume",
    "competition",
    "word_count",
    "char_count"
]

categorical_features = [
    "content_type",
    "main_intent"
]

feature_cols = numeric_features + categorical_features

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

y_train = train_df["declining_observed"]
y_test = test_df["declining_observed"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print("Features used:")
print(feature_cols)

print("\nNumber of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

Features used:
['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'days_since_last_update', 'content_age_days', 'avg_position', 'search_volume', 'competition', 'word_count', 'char_count', 'content_type', 'main_intent']

Number of numeric features: 10
Number of categorical features: 2


In [7]:
model.fit(X_train, y_train)

model_prob = model.predict_proba(X_test)[:, 1]

model_pred = (model_prob >= 0.5).astype(int)

print("Model trained successfully.")
print("Number of test predictions:", len(model_pred))

Model trained successfully.
Number of test predictions: 7115


In [8]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    return top_k.mean()
model_p20 = precision_at_k(model_prob, y_test, 20)
model_p50 = precision_at_k(model_prob, y_test, 50)

print("Model Precision@20:", round(model_p20, 3))
print("Model Precision@50:", round(model_p50, 3))

base_rate = y_test.mean()

print("Test-set base rate:", round(base_rate, 3))

Model Precision@20: 0.5
Model Precision@50: 0.62
Test-set base rate: 0.517


In [9]:
visibility_score = test_df["impression_tier"].isin(
    ["moderate", "good", "excellent"]
).astype(int)

stale_bonus = (
    test_df["days_since_last_update"] >= 91
).astype(int)

baseline_score = visibility_score + stale_bonus

baseline_p20 = precision_at_k(
    baseline_score,
    y_test,
    20
)

baseline_p50 = precision_at_k(
    baseline_score,
    y_test,
    50
)

print("Baseline Precision@20:", round(baseline_p20, 3))
print("Baseline Precision@50:", round(baseline_p50, 3))

Baseline Precision@20: 0.25
Baseline Precision@50: 0.44


In [14]:
comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Logistic Regression",
        "Test-set Base Rate"
    ],
    "Precision@20": [
        baseline_p20,
        model_p20,
        base_rate
    ],
    "Precision@50": [
        baseline_p50,
        model_p50,
        base_rate
    ]
})

print(comparison.round(3))

                Method  Precision@20  Precision@50
0      Week-4 Baseline         0.250         0.440
1  Logistic Regression         0.500         0.620
2   Test-set Base Rate         0.517         0.517


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model made 3,295 errors out of 7,115 test rows, giving an error rate of 46.3%. The largest error group was false positives: 1,988 pages were predicted as declining but were not declining according to the observed target. There were also 1,307 false negatives, where the model missed pages that did decline.

The wrong examples show that some pages are difficult to classify. Some false positives had moderate or good visibility but had recently been updated. Some false negatives declined despite having strong previous impressions or low visibility.

The largest absolute Logistic Regression coefficients were associated with content type and main intent. The strongest coefficient was for feedly article, followed by navigational intent and keyword article. These results show that the model relies strongly on content type and intent when making predictions. These are associations in the fitted model, not evidence that these features cause decline.

In [11]:
error_df = test_df.copy()

error_df["actual"] = y_test.values
error_df["predicted"] = model_pred
error_df["predicted_probability"] = model_prob

error_df["correct"] = (
    error_df["actual"] == error_df["predicted"]
)

print("Total test rows:", len(error_df))
print("Incorrect predictions:", (~error_df["correct"]).sum())
print(
    "Error rate:",
    round((~error_df["correct"]).mean(), 3)
)

print("\nError types:")
print(
    pd.crosstab(
        error_df["actual"],
        error_df["predicted"],
        rownames=["Actual"],
        colnames=["Predicted"]
    )
)

Total test rows: 7115
Incorrect predictions: 3295
Error rate: 0.463

Error types:
Predicted     0     1
Actual               
0          1452  1988
1          1307  2368


In [12]:
false_positives = error_df[
    (error_df["actual"] == 0) &
    (error_df["predicted"] == 1)
].copy()

print("Three false-positive examples:")
print(
    false_positives[
        [
            "content_id",
            "predicted_probability",
            "impressions_prev_30d",
            "days_since_last_update",
            "content_age_days",
            "avg_position",
            "impression_tier",
            "content_type"
        ]
    ].head(3).to_string(index=False)
)

false_negatives = error_df[
    (error_df["actual"] == 1) &
    (error_df["predicted"] == 0)
].copy()

print("\nThree false-negative examples:")
print(
    false_negatives[
        [
            "content_id",
            "predicted_probability",
            "impressions_prev_30d",
            "days_since_last_update",
            "content_age_days",
            "avg_position",
            "impression_tier",
            "content_type"
        ]
    ].head(3).to_string(index=False)
)

Three false-positive examples:
          content_id  predicted_probability  impressions_prev_30d  days_since_last_update  content_age_days  avg_position impression_tier       content_type
content_a5a2fbc76336               0.632263                    77                     103               238          39.8        moderate    keyword article
content_72c5c2d73e5a               0.503930                   818                      13               300          30.0        moderate    keyword article
content_55f75c034970               0.526476                   980                       8               140           6.4            good comparison article

Three false-negative examples:
          content_id  predicted_probability  impressions_prev_30d  days_since_last_update  content_age_days  avg_position impression_tier    content_type
content_a1fb4e703a9e               0.444459                  5915                      25               445          20.3            good keyword article
c

In [13]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "absolute_coefficient": np.abs(coefficients)
})

feature_importance = feature_importance.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Top 10 features:")
print(
    feature_importance[
        ["feature", "coefficient"]
    ].head(10).to_string(index=False)
)

Top 10 features:
                                  feature  coefficient
 categorical__content_type_feedly article    -0.784983
    categorical__main_intent_navigational    -0.576232
categorical__content_type_keyword article     0.489359
                numeric__content_age_days    -0.387496
                      numeric__word_count     0.235990
                      numeric__char_count    -0.228893
   categorical__main_intent_informational     0.213613
          numeric__days_since_last_update     0.194401
                 numeric__clicks_prev_30d    -0.158678
               numeric__sessions_prev_30d    -0.106236


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.